# Comparing prompt hacking with Qwen Math and Qwen vanilla models

### Qwen math + normal prompt

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

print(torch.cuda.is_available())

# Load model and tokenizer
model_id = "Qwen/Qwen2.5-Math-1.5B"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="bfloat16",
    trust_remote_code=True,
    attn_implementation="flash_attention_2" #<- uncomment on compatible GPU
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Generate answer
prompt1 = "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. Please integrate natural language reasoning with programs to solve the problem above, and put your final answer within \boxed{}."
prompt2 = "Let $x,y$ and $z$ be positive real numbers that satisfy the following system of equations: \n\\[\\log_2\\left({x \\over yz}\\right) = {1 \\over 2}\\]\n\\[\\log_2\\left({y \\over xz}\\right) = {1 \\over 3}\\]\n\\[\\log_2\\left({z \\over xy}\\right) = {1 \\over 4}\\]\nThen the value of $\\left|\\log_2(x^4y^3z^2)\\right|$ is $\\tfrac{m}{n}$ where $m$ and $n$ are relatively prime positive integers. Find $m+n$."

input_ids = tokenizer.apply_chat_template(
    [   
        {"role": "system", "content": prompt1},
        {"role": "user", "content": prompt2}
    ],
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
).to(model.device)

output = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.3,
    top_p=0.95,
#    min_p=0.15,
#    repetition_penalty=1.05,
    max_new_tokens=2048,
)

print(type(input_ids))
print(tokenizer.decode(output[0], skip_special_tokens=False))


True


config.json:   0%|          | 0.00/676 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<class 'torch.Tensor'>
<|im_start|>system
A conversation between User and Assistant. The user asks a question, and the Assistant solves it. Please integrate natural language reasoning with programs to solve the problem above, and put your final answer withinoxed{}.<|im_end|>
<|im_start|>user
Let $x,y$ and $z$ be positive real numbers that satisfy the following system of equations: 
\[\log_2\left({x \over yz}\right) = {1 \over 2}\]
\[\log_2\left({y \over xz}\right) = {1 \over 3}\]
\[\log_2\left({z \over xy}\right) = {1 \over 4}\]
Then the value of $\left|\log_2(x^4y^3z^2)\right|$ is $\tfrac{m}{n}$ where $m$ and $n$ are relatively prime positive integers. Find $m+n$.<|im_end|>
<|im_start|>assistant

<|im_start|>system
A conversation between User and Assistant. The user asks a question, and the Assistant solves it. Please integrate natural language reasoning with programs to solve the problem above, and put your final answer withinoxed{}.<|im_end|>
<|im_start|>user
Let $x,y$ and $z$ be po

### Result
The model actually doesn't use a think section here, interestingly

### Qwen math + hacked prompt

In [2]:
tokenizer.chat_template = "{{bos_token}}{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n" \
            + "<think>Python code blocks must use the format ```python to be executed. I can reason about the problem first and then generate a code block, get the response of the executed code to inform my answer. Let me begin reasoning:\n" \
            + "' }}{% endif %}"
print(tokenizer.chat_template)

{{bos_token}}{% for message in messages %}{{'<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>' + '
'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
<think>Python code blocks must use the format ```python to be executed. I can reason about the problem first and then generate a code block, get the response of the executed code to inform my answer. Let me begin reasoning:
' }}{% endif %}


In [3]:
input_ids = tokenizer.apply_chat_template(
    [   
        {"role": "system", "content": prompt1},
        {"role": "user", "content": prompt2}
    ],
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
).to(model.device)

output = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.3,
    top_p=0.95,
#    min_p=0.15,
#    repetition_penalty=1.05,
    max_new_tokens=2048,
)

print(type(input_ids))
print(tokenizer.decode(output[0], skip_special_tokens=False))


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


<class 'torch.Tensor'>
<|im_start|>system
A conversation between User and Assistant. The user asks a question, and the Assistant solves it. Please integrate natural language reasoning with programs to solve the problem above, and put your final answer withinoxed{}.<|im_end|>
<|im_start|>user
Let $x,y$ and $z$ be positive real numbers that satisfy the following system of equations: 
\[\log_2\left({x \over yz}\right) = {1 \over 2}\]
\[\log_2\left({y \over xz}\right) = {1 \over 3}\]
\[\log_2\left({z \over xy}\right) = {1 \over 4}\]
Then the value of $\left|\log_2(x^4y^3z^2)\right|$ is $\tfrac{m}{n}$ where $m$ and $n$ are relatively prime positive integers. Find $m+n$.<|im_end|>
<|im_start|>assistant
<think>Python code blocks must use the format ```python to be executed. I can reason about the problem first and then generate a code block, get the response of the executed code to inform my answer. Let me begin reasoning:

<|im_start|>system
A conversation between User and Assistant. The use

### Result
the model seamlessly integrates the think section into its response and generates a code block

### Qwen vanilla + normal prompt

In [5]:
model_id = "Qwen/Qwen3-1.7B"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="bfloat16",
    trust_remote_code=True,
    attn_implementation="flash_attention_2" #<- uncomment on compatible GPU
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Generate answer
prompt1 = "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. Please integrate natural language reasoning with programs to solve the problem above, and put your final answer within \boxed{}."
prompt2 = "Let $x,y$ and $z$ be positive real numbers that satisfy the following system of equations: \n\\[\\log_2\\left({x \\over yz}\\right) = {1 \\over 2}\\]\n\\[\\log_2\\left({y \\over xz}\\right) = {1 \\over 3}\\]\n\\[\\log_2\\left({z \\over xy}\\right) = {1 \\over 4}\\]\nThen the value of $\\left|\\log_2(x^4y^3z^2)\\right|$ is $\\tfrac{m}{n}$ where $m$ and $n$ are relatively prime positive integers. Find $m+n$."

input_ids = tokenizer.apply_chat_template(
    [   
        {"role": "system", "content": prompt1},
        {"role": "user", "content": prompt2}
    ],
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
).to(model.device)

output = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.3,
    top_p=0.95,
#    min_p=0.15,
#    repetition_penalty=1.05,
    max_new_tokens=2048,
)

print(type(input_ids))
print(tokenizer.decode(output[0], skip_special_tokens=False))


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

<class 'torch.Tensor'>
<|im_start|>system
A conversation between User and Assistant. The user asks a question, and the Assistant solves it. Please integrate natural language reasoning with programs to solve the problem above, and put your final answer withinoxed{}.<|im_end|>
<|im_start|>user
Let $x,y$ and $z$ be positive real numbers that satisfy the following system of equations: 
\[\log_2\left({x \over yz}\right) = {1 \over 2}\]
\[\log_2\left({y \over xz}\right) = {1 \over 3}\]
\[\log_2\left({z \over xy}\right) = {1 \over 4}\]
Then the value of $\left|\log_2(x^4y^3z^2)\right|$ is $\tfrac{m}{n}$ where $m$ and $n$ are relatively prime positive integers. Find $m+n$.<|im_end|>
<|im_start|>assistant

<|im_start|>system
A conversation between User and Assistant. The user asks a question, and the Assistant solves it. Please integrate natural language reasoning with programs to solve the problem above, and put your final answer withinoxed{}.<|im_end|>
<|im_start|>user
Let $x,y$ and $z$ be po

### Result
The model intializes its own think section

### Qwen vanilla + hacked prompt

In [6]:
model_id = "Qwen/Qwen3-1.7B"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="bfloat16",
    trust_remote_code=True,
    attn_implementation="flash_attention_2" #<- uncomment on compatible GPU
)

tokenizer.chat_template = "{{bos_token}}{% for message in messages %}{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>' + '\n'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant\n" \
            + "<think>Python code blocks must use the format ```python to be executed. I can reason about the problem first and then generate a code block, get the response of the executed code to inform my answer. Let me begin reasoning:\n" \
            + "' }}{% endif %}"
print(tokenizer.chat_template)

{{bos_token}}{% for message in messages %}{{'<|im_start|>' + message['role'] + '
' + message['content'] + '<|im_end|>' + '
'}}{% endfor %}{% if add_generation_prompt %}{{ '<|im_start|>assistant
<think>Python code blocks must use the format ```python to be executed. I can reason about the problem first and then generate a code block, get the response of the executed code to inform my answer. Let me begin reasoning:
' }}{% endif %}


In [7]:
input_ids = tokenizer.apply_chat_template(
    [   
        {"role": "system", "content": prompt1},
        {"role": "user", "content": prompt2}
    ],
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
).to(model.device)

output = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.3,
    top_p=0.95,
#    min_p=0.15,
#    repetition_penalty=1.05,
    max_new_tokens=2048,
)

print(type(input_ids))
print(tokenizer.decode(output[0], skip_special_tokens=False))


<class 'torch.Tensor'>
<|im_start|>system
A conversation between User and Assistant. The user asks a question, and the Assistant solves it. Please integrate natural language reasoning with programs to solve the problem above, and put your final answer withinoxed{}.<|im_end|>
<|im_start|>user
Let $x,y$ and $z$ be positive real numbers that satisfy the following system of equations: 
\[\log_2\left({x \over yz}\right) = {1 \over 2}\]
\[\log_2\left({y \over xz}\right) = {1 \over 3}\]
\[\log_2\left({z \over xy}\right) = {1 \over 4}\]
Then the value of $\left|\log_2(x^4y^3z^2)\right|$ is $\tfrac{m}{n}$ where $m$ and $n$ are relatively prime positive integers. Find $m+n$.<|im_end|>
<|im_start|>assistant
<think>Python code blocks must use the format ```python to be executed. I can reason about the problem first and then generate a code block, get the response of the executed code to inform my answer. Let me begin reasoning:

<|im_start|>system
A conversation between User and Assistant. The use

### Result
The model immediately terminates the think section, which is odd. It may be because a majority of its training comes from the enable_thinking=False scenario?


```
  {%- if add_generation_prompt %}\n
      {{- '<|im_start|>assistant\\n
  ' }}\n
      {%- if enable_thinking is defined and enable_thinking is false %}\n
          {{- '<think>\\n
  \\n
  </think>\\n
```

Which automatically includes the </think> token...

## Qwen vanilla + normal prompt + with enable thinking

In [13]:
model_id = "Qwen/Qwen3-1.7B"
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype="bfloat16",
    trust_remote_code=True,
    attn_implementation="flash_attention_2" #<- uncomment on compatible GPU
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

print(str(tokenizer.chat_template)[-400:])

input_ids = tokenizer.apply_chat_template(
    [   
        {"role": "system", "content": prompt1},
        {"role": "user", "content": prompt2}
    ],
    add_generation_prompt=True,
    enable_thinking=True,
    return_tensors="pt",
    tokenize=True,
).to(model.device)



Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

   {{- '\n</tool_response>' }}
        {%- if loop.last or (messages[loop.index0 + 1].role != "tool") %}
            {{- '<|im_end|>\n' }}
        {%- endif %}
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|im_start|>assistant\n' }}
    {%- if enable_thinking is defined and enable_thinking is false %}
        {{- '<think>\n\n</think>\n\n' }}
    {%- endif %}
{%- endif %}


In [14]:
output = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.3,
    top_p=0.95,
#    min_p=0.15,
#    repetition_penalty=1.05,
    max_new_tokens=2048,
)

print(type(input_ids))
print(tokenizer.decode(output[0], skip_special_tokens=False))


<class 'torch.Tensor'>
<|im_start|>system
A conversation between User and Assistant. The user asks a question, and the Assistant solves it. Please integrate natural language reasoning with programs to solve the problem above, and put your final answer withinoxed{}.<|im_end|>
<|im_start|>user
Let $x,y$ and $z$ be positive real numbers that satisfy the following system of equations: 
\[\log_2\left({x \over yz}\right) = {1 \over 2}\]
\[\log_2\left({y \over xz}\right) = {1 \over 3}\]
\[\log_2\left({z \over xy}\right) = {1 \over 4}\]
Then the value of $\left|\log_2(x^4y^3z^2)\right|$ is $\tfrac{m}{n}$ where $m$ and $n$ are relatively prime positive integers. Find $m+n$.<|im_end|>
<|im_start|>assistant
<think>
Okay, let's try to solve this problem step by step. So, we have three equations involving logarithms with base 2, and we need to find the value of |log₂(x⁴y³z²)|, which is m/n and then find m + n. 

First, let me write down the given equations again to make sure I have them right:

1. 